In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.data import Subset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.linear = torch.nn.Linear(100, 1024*4*4)
        self.deconv1 = nn.Sequential(nn.ConvTranspose2d(1024, 512, 4, 2, 1, bias=False), nn.BatchNorm2d(512), nn.ReLU(True))
        self.deconv2 = nn.Sequential(nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True))
        self.deconv3 = nn.Sequential(nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True))
        self.deconv4 = nn.Sequential(nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False), nn.Tanh())

    def forward(self, x):
        x = self.linear(x)
        x = x.view(-1, 1024, 4, 4)
        x = self.deconv1(x)
        x = self.deconv2(x)
        x = self.deconv3(x)
        x = self.deconv4(x)
        return x

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(3, 128, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, inplace=True))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True))
        self.conv3 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1, bias=False), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, inplace=True))
        self.conv4 = nn.Sequential(nn.Conv2d(512, 1024, 4, 2, 1, bias=False), nn.BatchNorm2d(1024), nn.LeakyReLU(0.2, inplace=True))
        self.output = nn.Sequential(nn.Linear(1024*4*4, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = x.view(-1, 1024*4*4)
        return self.output(x)

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np
def generate_images(generator, test_noise, num_images , epoch, batch):
    with torch.no_grad():
        test_images = generator(test_noise).detach().cpu()
    plt.figure(figsize=(8, 8))
    plt.axis("off")
    plt.title(f"Images at Epoch {epoch}, Batch {batch}")
    plt.imshow(np.transpose(vutils.make_grid(test_images[:num_images], padding=2, normalize=True), (1, 2, 0)))
    plt.show()

def init_weights(m):
    classname = m.__class__.__name__
    if 'Conv' in classname or 'BatchNorm' in classname:
        m.weight.data.normal_(0.00, 0.02)

def noise(size):
    n = torch.randn(size, 100)
    return n.to(device)

def real_data_target(size):
    data = torch.ones(size, 1)
    return data.to(device)

def fake_data_target(size):
    data = torch.zeros(size, 1)
    return data.to(device)

In [ ]:
def train_discriminator(discriminator, optimizer, real_data, fake_data, criterion):
    optimizer.zero_grad()
    real = discriminator(real_data)
    real_error = criterion(real, real_data_target(real_data.size(0)))
    real_error.backward()
    fake = discriminator(fake_data.detach())
    fake_error = criterion(fake, fake_data_target(fake_data.size(0)))
    fake_error.backward()
    optimizer.step()
    
    return real_error + fake_error, real, fake

def train_generator(generator, discriminator, optimizer, fake_data, criterion):
    optimizer.zero_grad()
    prediction = discriminator(fake_data)
    error = criterion(prediction, real_data_target(prediction.size(0)))
    error.backward()
    optimizer.step()
    return error

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = Generator().to(device)
discriminator = Discriminator().to(device)
generator.apply(init_weights)
discriminator.apply(init_weights)

# Optimizers
lr = 0.0002
gen_optimizer = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
disc_optimizer = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
criterion = nn.BCELoss()

transform = transforms.Compose(
        [
            transforms.Resize(64),
            transforms.ToTensor(),
            transforms.Normalize((.5, .5, .5), (.5, .5, .5))
        ])
dataset = datasets.CIFAR10(root='./data', download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

data, targets = dataset.data, dataset.targets
label_datasets = {}

for label in range(10):
    indices = [i for i, t in enumerate(targets) if t == label]
    label_datasets[label] = Subset(dataset, indices)

epochs = 100
for class_num in range(10):
    print(f"Training for class {class_num}")
    label_data = label_datasets[class_num]
    dataloader = DataLoader(label_data, batch_size=64, shuffle=True)
    for epoch in range(epochs):
        count = 0
        for real_data, _ in dataloader:
            real_data = real_data.to(device)
            real_data_size = real_data.size(0)
            fake_data = generator(noise(real_data_size)).to(device)
            d_error, d_real, d_fake = train_discriminator(discriminator, disc_optimizer, real_data, fake_data, criterion)
            fake_data = generator(noise(real_data_size)).to(device)
            g_error = train_generator(generator, discriminator, gen_optimizer, fake_data, criterion)
            if count % 50 == 0:
                print(f"Epoch: {epoch}, Batch: {count}, D_error: {d_error.item()}, G_error: {g_error.item()}")
                generate_images(generator, noise(16), 16, epoch, count)
            count += 1
    torch.save(generator.state_dict(), f"generator_{class_num}.pth")
    torch.save(discriminator.state_dict(), f"discriminator_{class_num}.pth")
